# Architecture-aware Gauge Playground

## Goal

在固定 codec 下只改变严格正交 latent 坐标 `y=A(z)`。这个 notebook 暴露
最简接口 `A(z)`、`A_inv(y)`，先验证重建和扩散噪声路径严格等价，再用
paired tiny probe 快速检查 identity 附近是否可能存在 headroom。

这里的 quick probe 只是 go/no-go smoke，不是 FID 或论文结论。

## Setup

### 1. Imports

数据集从 `/data/shared` 读取；模型与本机结果由仓库软链接指向
`$HOME/data/eqvae`。默认全程 fp32。

In [ ]:
from pathlib import Path
import sys
import torch
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.architecture_gauge import (
    CodecDataConfig, GaugeSpec, ProbeConfig, ProbeTrainingConfig,
    add_identity_ratios, exact_equivalence_table, finite_difference_headroom,
    make_gauge, plot_learning_curves, prepare_codec_data,
    reconstruction_equivalence_table, run_probe_grid,
    visualize_gauge_roundtrip, visualize_latent_pca,
)

### 2. Parameters

日常只改这个单元。`MODEL_KEY` 可选 `rae_dinov2`、`rae_mae`、
`rae_siglip2`、`sdvae`、`eqvae`。`A_KIND` 可选 `identity`、`roll`、
`channel_givens`、`fourier_allpass`、`block_haar`。

In [ ]:
MODEL_KEY = "rae_dinov2"
DATASET_NAME = "imagenet_parquet"
DATASET_PATH = "/data/shared/imagenet-1k"
TRAIN_SPLIT, VAL_SPLIT = "train", "validation"
TRAIN_COUNT, VAL_COUNT = 16, 8
IMAGE_SIZE = 256
DEVICE = "cuda:0"
SEED = 0

# 当前 A。all-pass 的 radius 控制主要混合距离，strength 控制强度。
A_SPEC = GaugeSpec(
    name="my_A",
    kind="fourier_allpass",
    strength=0.45,
    radius=2,
    seed=SEED,
)

# identity 邻域的对称方向，用于估计 directional gradient / curvature。
HEADROOM_DELTA = 0.25
COMPARE_GAUGES = [
    GaugeSpec("identity"),
    GaugeSpec("allpass_plus", kind="fourier_allpass", strength=+HEADROOM_DELTA, radius=1),
    GaugeSpec("allpass_minus", kind="fourier_allpass", strength=-HEADROOM_DELTA, radius=1),
    GaugeSpec("channel_0.5", kind="channel_givens", strength=0.5, seed=SEED),
]

RUN_QUICK_PROBE = True
QUICK_STEPS = 8

## Steps

### 3. Load frozen codec and disjoint data

ImageNet 默认直接使用 train/validation 两个官方 split，因此 quick probe
不会把 validation 图像用于更新。

In [ ]:
data = prepare_codec_data(CodecDataConfig(
    dataset_name=DATASET_NAME,
    dataset_path=DATASET_PATH,
    train_split=TRAIN_SPLIT,
    val_split=VAL_SPLIT,
    train_count=TRAIN_COUNT,
    val_count=VAL_COUNT,
    image_size=IMAGE_SIZE,
    model_key=MODEL_KEY,
    device=DEVICE,
    seed=SEED,
))

print("train images:", tuple(data.train_images.shape), "train z:", tuple(data.train_latents.shape))
print("val images:  ", tuple(data.val_images.shape), "val z:  ", tuple(data.val_latents.shape))
print("latent RMS scale:", data.latent_scale)

### 4. Minimal `A` interface

In [ ]:
active_gauge = make_gauge(A_SPEC)

def A(z):
    return active_gauge.forward(z)

def A_inv(y):
    return active_gauge.inverse(y)

x = data.val_images
z = data.val_latents.to(data.device)
y = A(z)
z_roundtrip = A_inv(y)

print("A:", A_SPEC)
print("max |A_inv(A(z))-z|:", float((z_roundtrip - z).abs().max()))

### 5. Exact equivalence gate

`inverse/norm/distance/noise` 应接近 fp32 数值误差。all-pass 还应保持总 PSD；
`block_haar` 是正交的，但故意不保持逐频率 PSD。

In [ ]:
exact = exact_equivalence_table(data.val_latents, [A_SPEC, *COMPARE_GAUGES], device=DEVICE)
recon_exact = reconstruction_equivalence_table(data, [A_SPEC, *COMPARE_GAUGES], count=2)
display(exact.round(8))
display(recon_exact.round(8))

assert exact[["inverse_rel_l2", "norm_rel_error", "paired_noise_rel_error"]].to_numpy().max() < 1e-5
assert recon_exact["image_mean_abs_error"].max() < 1e-4

### 6. Visualize the same information in two coordinates

`D(Az)` 只用于展示 decoder 收到错误坐标时会发生什么，不属于 AGF 方法。
方法中的 decoder 输入始终是 `A_inv(y)`。

In [ ]:
visualize_gauge_roundtrip(data, A_SPEC, count=min(3, VAL_COUNT));
visualize_latent_pca(data.val_latents, A_SPEC, count=min(4, VAL_COUNT));

### 7. Optional paired headroom probe

同一 probe 下的 `identity/+delta/-delta` 使用相同样本、噪声、时间、初始化
和步数。`g != 0` 只表示该 tiny probe 的 identity 可能不是 stationary point；
至少 3 个 seed 与正式生成迁移成功后才能称为 H2。

In [ ]:
if RUN_QUICK_PROBE:
    quick_training = ProbeTrainingConfig(
        steps=QUICK_STEPS,
        eval_steps=(0, QUICK_STEPS // 2, QUICK_STEPS),
        batch_size=min(4, TRAIN_COUNT),
        eval_batches=2,
        time_bins=4,
        seed=SEED,
    )
    quick_probes = [
        ProbeConfig("local_rf9", kind="local", hidden=16, depth=2),
        ProbeConfig("global_attn", kind="global", hidden=16, depth=1, heads=4),
    ]
    quick_runs, quick_history, quick_bins = run_probe_grid(
        data.train_latents,
        data.val_latents,
        COMPARE_GAUGES[:3],
        quick_probes,
        quick_training,
        seeds=(SEED,),
        device=DEVICE,
        latent_scale=data.latent_scale,
    )
    display(add_identity_ratios(quick_history).round(5))
    display(finite_difference_headroom(
        quick_history,
        plus_gauge="allpass_plus",
        minus_gauge="allpass_minus",
        delta=HEADROOM_DELTA,
    ).round(5))
    plot_learning_curves(quick_history);
else:
    print("RUN_QUICK_PROBE=False: exact and visual checks completed.")

## Checks

- Exact gate 失败：先修 `A/A_inv`、noise pairing 或 codec normalization。
- 只有坏坐标、没有坐标优于 identity：只支持 H1，不支持质量方法。
- 单 seed tiny probe 的小差异：只能用于排查代码和决定下一组方向。
- `D(Az)` 很差但 `D(A_inv(Az))` 精确：说明 decoder 只接受原坐标，符合设计。

## Next Steps

用 `gauge_mechanism_routing.ipynb` 检查差异究竟来自 locality、有限训练预算、
decoder 放大还是 time-bin 冲突。